# Acquire + deskew on the fly

Runs a LightSheetManager galvo swept acquisition and **deskews each volume on the GPU
as soon as that volume finishes writing** — while the next one is still being acquired.

Both halves run in the same environment (miniforge3 base): `pycromanager` drives the
acquisition, `pyclesperanto_prototype` does the deskew on the RTX 3090.

**Why per-volume and not per-frame:** `deskew_y` shears the sweep axis into Y, so it
needs every Z plane of a volume before it can produce any output. The earliest
possible moment to deskew is when a volume's last slice lands — which is exactly what
this does. For a time series that means volume *N* is deskewing while *N+1* sweeps.

**Order:** connect → parameters → apply → run (acquire+deskew) → inspect → close.

## 1. Connect

In [ ]:
import json, math, time
from pathlib import Path

import numpy
import matplotlib.pyplot as plt
import tifffile as tff
import pyclesperanto_prototype as cle
from pyclesperanto_prototype._tier8._affine_transform import _determine_translation_and_bounding_box
from ndtiff import Dataset

from pycromanager import Core
from lsm_pycromanager import LightSheetManager

core = Core()
lsm_mgr = LightSheetManager()
lsm = lsm_mgr.open()

print('GPU        :', cle.get_device())
print('Core camera:', core.get_camera_device())
print('LSM camera :', lsm.devices().first_active_camera_name())

## 2. Parameters

In [ ]:
# ================= ACQUISITION =================
SLICES_PER_VIEW    = 101     # planes per volume
SLICE_STEP_UM      = 1.0     # um between planes (the raw galvo step)
MINIMIZE_SLICE_PERIOD = True # let LSM fit the period to the exposure

# ---- Camera ROI ----
# int -> centred square; (w, h) -> centred rectangle. None = full frame.
# Sensor is 1200x1200 binned (2400 unbinned at 2x2).
#
# IMPORTANT: ROI *height* caps the exposure. In PSEUDO_OVERLAP the camera reads
# out row by row, so the slice budget is roughly:
#       budget_ms = roi_height * LINE_SCAN_US / 1000
# and MicroManager rejects the acquisition if
#       cameraExposure + readout > budget
# where cameraExposure = SAMPLE_EXPOSURE_MS + ~8.5 ms of scan/camera delay.
# Cropping 1200 -> 600 rows halves the budget from ~49 ms to ~24.5 ms, which is
# why a 20 ms exposure that worked at full frame fails at 600 px.
ROI_SIZE_PX = None           # full frame - the config that works.
                             # Cropping shrinks the exposure budget (see above):
                             # 600 rows caps you at 15.75 ms, which is why the
                             # 20 ms exposure failed. (600, 720) would allow 20 ms.
ROI_CAMERAS = ['Kinetix22-1', 'Kinetix22-2']

LINE_SCAN_US       = 40.83   # per binned row, at 100MHz 12bit / 2x2 binning
CAMERA_DELAY_MS    = 8.5     # cameraExposure - sampleExposure
READOUT_MS         = 0.25

SAMPLE_EXPOSURE_MS = 20.0    # full frame budget is 49 ms, so 20 ms is fine

# ---- Time series ----
USE_TIME_POINTS = True
NUM_TIME_POINTS = 5
TIME_INTERVAL_S = 7.0        # seconds between volumes (0 is rejected by LSM)

USE_CHANNELS  = False
CHANNEL_GROUP = 'Channel+Filter'
CHANNELS      = ['488+561']

SAVE_DIRECTORY   = r'C:\Users\aifadmin\Desktop\TEST'
SAVE_NAME_PREFIX = 'zstack'
# Save format is left to the LightSheetManager GUI. The live deskew reads ND-TIFF.

# ================= LASERS =================
LASER_VOLTAGES = {
    '405': None, '445': None, '488': 1.0,
    '515': None, '561': 1.0,  '638': 1.0,
}

# ================= DESKEW =================
# Matches ASI_NEW_soLISH_20241219_lowmem.ipynb
lightsheet_angle_in_degrees = 40.5
time_vs_track  = 1
scaling_factor = 1
max_width      = 2048
dual_camera    = 1
galvo_vs_stage = 1

# LSM interleaves cameras across the channel axis ([cam1, cam2, cam1, cam2]),
# unlike the OME-TIFF notebooks which assume camera-major order, so the flip is
# decided by the recorded camera NAME rather than by channel index.
FLIP_CAMERA = 'Kinetix22-1'

selected_channels = 0
channelsList      = [0, 2, 3]
selected_positions = 0
posList            = [1, 2]
timepoints_start = 0
timepoints_range = 0

Z_max_projections = 1
Y_max_projections = 0
data3D_save       = 1
data_type         = numpy.uint16

VOXEL_XY_OVERRIDE = None

deskewing_angle_in_degrees = (lightsheet_angle_in_degrees if galvo_vs_stage
                              else 180 - lightsheet_angle_in_degrees)
corr_value = numpy.sin(deskewing_angle_in_degrees / 180 * math.pi)
voxel_z_um = SLICE_STEP_UM / corr_value

# Volume depth comes from the settings, never from the axes observed on disk —
# during acquisition the z axis grows as slices land, so a half-written volume
# would otherwise look complete.
EXPECTED_NZ = SLICES_PER_VIEW

# ---- exposure budget preflight ----
_roi = ROI_SIZE_PX
_h = None if _roi is None else (_roi[1] if isinstance(_roi, (tuple, list)) else _roi)
if _h:
    budget_ms  = _h * LINE_SCAN_US / 1000.0
    max_sample = budget_ms - READOUT_MS - CAMERA_DELAY_MS
else:
    budget_ms, max_sample = None, None

print(f'Volume       : {SLICES_PER_VIEW} x {SLICE_STEP_UM} um '
      f'= {SLICES_PER_VIEW * SLICE_STEP_UM:.1f} um deep')
print(f'ROI          : {_roi if _roi else "full frame"}'
      f'{f"  -> slice budget {budget_ms:.1f} ms" if budget_ms else ""}')
print(f'Exposure     : {SAMPLE_EXPOSURE_MS} ms'
      f'{f"  (max {max_sample:.2f} ms for this ROI)" if max_sample else ""}')
print(f'Time points  : {NUM_TIME_POINTS if USE_TIME_POINTS else 1}'
      f'{f" every {TIME_INTERVAL_S} s" if USE_TIME_POINTS else ""}')
print(f'Deskew angle : {deskewing_angle_in_degrees} deg   -> voxel Z = {voxel_z_um:.4f} um')
print(f'Flip camera  : {FLIP_CAMERA if dual_camera else "(dual_camera off)"}')

if max_sample is not None and SAMPLE_EXPOSURE_MS > max_sample:
    need_h = int(numpy.ceil((SAMPLE_EXPOSURE_MS + CAMERA_DELAY_MS + READOUT_MS)
                            / LINE_SCAN_US * 1000))
    print(f'\n*** SAMPLE_EXPOSURE_MS={SAMPLE_EXPOSURE_MS} EXCEEDS the '
          f'{max_sample:.2f} ms budget for a {_h}-row ROI.')
    print(f'    MicroManager will refuse the acquisition. Either:')
    print(f'      - lower SAMPLE_EXPOSURE_MS to <= {max_sample:.2f}, or')
    print(f'      - use a taller ROI: ROI_SIZE_PX = ({_roi[0] if isinstance(_roi,(tuple,list)) else _roi}, {need_h})')

## 3. Apply acquisition settings

In [ ]:
def _deep_merge(base, patch):
    for k, v in patch.items():
        if isinstance(v, dict) and isinstance(base.get(k), dict):
            _deep_merge(base[k], v)
        else:
            base[k] = v
    return base


def patch_settings(lsm, patch):
    """Apply every change in ONE update_settings call.

    Splitting this into a builder push followed by a JSON patch leaves the
    plugin in an intermediate state that gets validated against the previous
    period/ROI — that is what raised the exposure error, and it also caused the
    time-point settings to be silently dropped. One atomic update avoids both.
    """
    settings = lsm.acquisitions().settings()
    cls = settings.get_class()
    if 'java.lang.Class' not in cls._interfaces:
        cls._interfaces.append('java.lang.Class')
    j = _deep_merge(json.loads(settings.to_pretty_json()), patch)
    lsm.acquisitions().update_settings(settings.from_json(json.dumps(j), cls))


# ---------- camera ROI ----------
def full_frame(cam):
    """Full binned frame size, from the chip dimensions and current binning."""
    xdim = int(float(core.get_property(cam, 'X-dimension')))
    ydim = int(float(core.get_property(cam, 'Y-dimension')))
    b = int(str(core.get_property(cam, 'Binning')).split('x')[0])
    return xdim // b, ydim // b


def apply_roi(cam, size):
    """Centre an ROI; int -> square, (w,h) -> rect, None -> full frame.

    Computes the centre from the chip dimensions rather than clearing first:
    clear_roi() takes no camera label (so it would force switching the Core
    camera) and restoring the full frame costs ~7 s per camera.
    """
    fw, fh = full_frame(cam)
    if size is None:
        target = (0, 0, fw, fh)
    else:
        w, h = size if isinstance(size, (tuple, list)) else (size, size)
        target = ((fw - w) // 2, (fh - h) // 2, w, h)
    r = core.get_roi(cam)
    if (r.x, r.y, r.width, r.height) == target:
        return False                       # already correct, skip the reconfigure
    core.set_roi(cam, *(int(v) for v in target))
    return True


for cam in ROI_CAMERAS:
    try:
        t_roi = time.time()
        changed = apply_roi(cam, ROI_SIZE_PX)
        r = core.get_roi(cam)
        print(f'ROI {cam:14s}: x={r.x} y={r.y} w={r.width} h={r.height}'
              f'  ({"set in %.2fs" % (time.time()-t_roi) if changed else "already set"})')
    except Exception as e:
        print(f'ROI {cam:14s}: FAILED ({e})')

# ---------- lasers / shutter ----------
for line, volts in LASER_VOLTAGES.items():
    if volts is None:
        continue
    try:
        core.set_property(line, 'Voltage', str(float(volts)))
    except Exception as e:
        print(f'  laser {line}: FAILED ({e})')

# LightSheetManager gates the lasers through the shutter during its sweep, so
# auto-shutter MUST be on. With it off the lasers never fire and the volume is
# just read noise.
core.set_auto_shutter(True)

print('Laser voltages :', {k: core.get_property(k, 'Voltage')
                           for k in LASER_VOLTAGES if LASER_VOLTAGES[k] is not None})
print('Auto-shutter   :', core.get_auto_shutter(), '(must be True for LSM)')
print('PLogic channel :', core.get_property('PLogic:E:36', 'OutputChannel'))

# ---------- acquisition settings (single atomic update) ----------
patch = {
    'volume_': {'slicesPerView_': int(SLICES_PER_VIEW),
                'sliceStepSize_': float(SLICE_STEP_UM)},
    'slice_':  {'sampleExposure_': float(SAMPLE_EXPOSURE_MS),
                'periodMinimized_': bool(MINIMIZE_SLICE_PERIOD)},
    'useTimePoints_': bool(USE_TIME_POINTS),
    'numTimePoints_': int(NUM_TIME_POINTS),
    'timePointInterval_': float(TIME_INTERVAL_S),
    'saveDuringAcq_': True,
    'saveDirectory_': SAVE_DIRECTORY,
    'saveNamePrefix_': SAVE_NAME_PREFIX,
}
if USE_CHANNELS:
    patch['channels_'] = {
        'enabled_': True, 'mode_': 'VOLUME', 'group_': CHANNEL_GROUP,
        'groups_': {CHANNEL_GROUP: [
            {'useChannel_': True, 'group_': CHANNEL_GROUP, 'name_': ch, 'offset_': 0.0}
            for ch in CHANNELS]},
    }
patch_settings(lsm, patch)

s = json.loads(lsm.acquisitions().settings().to_pretty_json())
vol, tim, sl = s['volume_'], s['timing_'], s['slice_']
n, d = vol['slicesPerView_'], tim['sliceDuration_']
nt = s['numTimePoints_'] if s['useTimePoints_'] else 1
print()
print(f"slicesPerView   : {n}")
print(f"sliceStepSize   : {vol['sliceStepSize_']} um")
print(f"sampleExposure  : {sl['sampleExposure_']} ms")
print(f"periodMinimized : {sl['periodMinimized_']}")
print(f"sliceDuration   : {d} ms")
print(f"cameraExposure  : {tim['cameraExposure_']} ms")
print(f"timePoints      : {nt}  (interval {s['timePointInterval_']} s)")
print(f"saveMode        : {s['saveMode_']}")
print(f"\n-> {n * d / 1000:.2f} s per volume x {nt} = "
      f"{n * d / 1000 * nt + max(0, nt - 1) * s['timePointInterval_']:.1f} s total")

problems = []
if int(n) != EXPECTED_NZ:
    problems.append(f"slicesPerView {n} != EXPECTED_NZ {EXPECTED_NZ} "
                    f"— set SLICES_PER_VIEW={int(n)} and re-run the parameters cell")
if not sl['periodMinimized_'] and MINIMIZE_SLICE_PERIOD:
    problems.append("periodMinimized did not stick — the exposure error will return")
if int(nt) != (NUM_TIME_POINTS if USE_TIME_POINTS else 1):
    problems.append(f"timePoints {nt} != requested {NUM_TIME_POINTS}")
if s['saveMode_'] != 'ND_TIFF':
    problems.append("saveMode is not ND_TIFF — the deskew reader expects ND-TIFF")

# Exposure must fit the ROI's line-scan budget or MM refuses at acquisition start
_r = core.get_roi(ROI_CAMERAS[0])
_budget = _r.height * LINE_SCAN_US / 1000.0
if tim['cameraExposure_'] + READOUT_MS > _budget:
    problems.append(
        f"cameraExposure {tim['cameraExposure_']} + {READOUT_MS} exceeds the "
        f"{_budget:.1f} ms line-scan budget for a {_r.height}-row ROI - "
        f"lower SAMPLE_EXPOSURE_MS or use a taller ROI")
else:
    print(f"exposure budget : {tim['cameraExposure_'] + READOUT_MS:.2f} / "
          f"{_budget:.1f} ms  ({_r.height} rows)  OK")
for p in problems:
    print(f"\nWARNING: {p}")

## 4. Deskew engine

GPU buffers are built lazily from the first frame (its size isn't known until then).
Reading is one 3D stack at a time — never the whole dataset.

In [ ]:
class Deskewer:
    """Holds the GPU buffers and deskews one (position, time, channel) volume."""

    def __init__(self, n_z, height, width, vxy, vz):
        self.n_z, self.h, self.w = n_z, height, width
        self.vx = self.vy = float(vxy)
        self.vz = float(vz)

        transform = cle.AffineTransform3D()
        transform._deskew_y(angle_in_degrees=deskewing_angle_in_degrees,
                            voxel_size_x=self.vx, voxel_size_y=self.vy,
                            voxel_size_z=self.vz, scale_factor=scaling_factor)
        probe = cle.create((n_z, height, width))
        self.new_size, _, _ = _determine_translation_and_bounding_box(probe, transform)
        self.mw = min(max_width, self.new_size[2])

        self.deskewed_ = cle.create([self.new_size[0], self.new_size[1], self.mw])
        self.max_z_    = cle.create([self.new_size[1], self.mw])
        self.max_y_    = cle.create([self.mw, self.new_size[0]])
        self.data_rot  = cle.create([n_z, height, self.mw])

        self.deskewed = numpy.zeros(self.new_size, dtype=data_type)
        self.max_z    = numpy.zeros(self.new_size[1:3], dtype=data_type)
        self.max_y    = numpy.zeros([self.new_size[2], self.new_size[0]], dtype=data_type)

    def run(self, image3D, flip):
        if flip:
            for zz in range(len(image3D)):
                image3D[zz] = numpy.fliplr(image3D[zz])

        num_steps = int(numpy.ceil(image3D.shape[2] / self.mw))
        delta = self.mw
        for ll in range(num_steps):
            lo, hi = ll * self.mw, ll * self.mw + self.mw
            hi_inv = image3D.shape[2] - ll * self.mw
            lo_inv = hi_inv - self.mw
            if hi > image3D.shape[2]:
                hi, lo_inv, delta = image3D.shape[2], 0, image3D.shape[2] - lo

            if galvo_vs_stage == 0:
                cle.rotate(image3D[:, :, lo:hi], self.data_rot,
                           angle_around_z_in_degrees=180, rotate_around_center=True)
                src = self.data_rot
            else:
                src = image3D[:, :, lo:hi]

            cle.deskew_y(src, self.deskewed_,
                         angle_in_degrees=deskewing_angle_in_degrees,
                         voxel_size_x=self.vx, voxel_size_y=self.vy, voxel_size_z=self.vz,
                         linear_interpolation=True, scale_factor=scaling_factor)
            self.deskewed[:, :, lo_inv:hi_inv] = self.deskewed_[:, :, 0:delta]

            if Z_max_projections:
                cle.maximum_z_projection(self.deskewed_, self.max_z_)
                self.max_z[:, lo_inv:hi_inv] = self.max_z_[:, 0:delta]
            if Y_max_projections:
                cle.maximum_y_projection(self.deskewed_, self.max_y_)
                self.max_y[lo_inv:hi_inv, :] = self.max_y_[0:delta, :]

        return self.deskewed, self.max_z, self.max_y


def axes_of(ds):
    return {k: sorted(v) for k, v in ds.axes.items()}


def coords_for(ax, p, t, c, z=None):
    d = {}
    if 'position' in ax: d['position'] = p
    if 'time'     in ax: d['time'] = t
    if 'channel'  in ax: d['channel'] = c
    if z is not None:    d['z'] = z
    return d


def camera_map(ds, ax, p, t):
    """channel index -> camera name, from the per-image metadata.

    LSM interleaves cameras across channels ([cam1, cam2, cam1, cam2]) whereas the
    OME-TIFF notebooks assume camera-major order ([cam1, cam1, cam2, cam2]).
    Reading the recorded Camera name is order-independent and can't get this wrong.
    """
    out = {}
    for c in ax.get('channel', [0]):
        try:
            out[c] = ds.read_metadata(**coords_for(ax, p, t, c, 0)).get('Camera')
        except Exception:
            out[c] = None
    return out


def volume_ready(ds, ax, p, t, c, n_z=None):
    """True once ALL expected z planes of this (p,t,c) are on disk.

    n_z comes from the acquisition settings, never from ds.axes — during a live
    acquisition the observed z axis grows as slices arrive, so checking against
    it would report a half-written volume as complete.
    """
    n_z = EXPECTED_NZ if n_z is None else n_z
    # Fast gate: slices are written in order, so if the last one is missing the
    # volume can't be complete. Saves ~100 index lookups per poll per channel.
    if not ds.has_image(**coords_for(ax, p, t, c, n_z - 1)):
        return False
    for z in range(n_z):
        if not ds.has_image(**coords_for(ax, p, t, c, z)):
            return False
    return True


def read_volume(ds, ax, p, t, c, n_z, h, w, dtype):
    out = numpy.empty((n_z, h, w), dtype=dtype)
    for z in range(n_z):
        out[z] = ds.read_image(**coords_for(ax, p, t, c, z))
    return out


print(f'deskew engine ready  (expecting {EXPECTED_NZ} slices per volume)')
print(f'flip camera        : {FLIP_CAMERA}')

## 5. Run — acquire and deskew together

Starts the acquisition, then watches the dataset. Each volume is deskewed the moment
its last slice lands, so processing overlaps the next sweep.

In [ ]:
import contextlib, io

POLL_S    = 0.5
TIMEOUT_S = 3600
QUIET_S   = 5.0      # index unchanged this long => writer has finished

# Reading an ND-TIFF while LightSheetManager is still appending to it takes file
# handles on the stack it is actively writing. On this rig that coincided with
# acquisition failures ("Expected images did not arrive in circular buffer") and
# a UI hang, so the safe default is to deskew only AFTER the writer goes quiet.
# Set True only for long time series where overlapping matters, and watch for
# acquisition errors if you do.
DESKEW_DURING_ACQ = False


@contextlib.contextmanager
def quiet():
    """Silence ndtiff's 'Reading index...' progress spam."""
    with contextlib.redirect_stdout(io.StringIO()):
        yield


def deskew_pending(run_dir, outdir, state):
    """Deskew every complete, not-yet-done volume. Returns True if any remain pending."""
    pending = False
    try:
        with quiet():
            ds = Dataset(str(run_dir))
    except Exception:
        return True

    try:
        ax = axes_of(ds)
        for p in ax.get('position', [0]):
            if selected_positions and p not in posList:
                continue
            for t in ax.get('time', [0]):
                if t < timepoints_start:
                    continue
                if timepoints_range and t >= timepoints_start + timepoints_range:
                    continue

                if state['cams'] is None:
                    state['cams'] = camera_map(ds, ax, p, t)
                    print('  channel -> camera:',
                          ', '.join(f'C{k}={v}' for k, v in state['cams'].items()))

                for c in ax.get('channel', [0]):
                    if selected_channels and c not in channelsList:
                        continue
                    if (p, t, c) in state['done']:
                        continue
                    if not volume_ready(ds, ax, p, t, c):
                        pending = True
                        continue

                    if state['dsk'] is None:
                        c0 = coords_for(ax, p, t, c, 0)
                        probe = ds.read_image(**c0)
                        md = ds.read_metadata(**c0)
                        vxy = VOXEL_XY_OVERRIDE or md.get('PixelSizeUm')
                        if not vxy:
                            raise ValueError('No PixelSizeUm — set VOXEL_XY_OVERRIDE')
                        state['dsk'] = Deskewer(EXPECTED_NZ, probe.shape[0],
                                                probe.shape[1], vxy, voxel_z_um)
                        d = state['dsk']
                        print(f'  voxel xy={d.vx} um  z={d.vz:.4f} um  nz={d.n_z}'
                              f'  -> output {d.new_size}')

                    dsk = state['dsk']
                    flip = bool(dual_camera and state['cams'].get(c) == FLIP_CAMERA)

                    tv = time.time()
                    img = read_volume(ds, ax, p, t, c, dsk.n_z, dsk.h, dsk.w, ds.dtype)
                    vol3d, mz, my = dsk.run(img, flip)

                    stem = (f'{run_dir.name}.deskewed_TRUE'
                            f'_Pos{str(p).rjust(2, "0")}'
                            f'_CH{str(c).rjust(2, "0")}'
                            f'_T{str(t).rjust(3, "0")}.tif')
                    if data3D_save:
                        tff.imwrite(str(outdir / stem), vol3d.astype(data_type))
                    if Z_max_projections:
                        tff.imwrite(str(outdir / 'MaxProject_Z' / f'MAX_{stem}'),
                                    mz.astype(data_type))
                    if Y_max_projections:
                        tff.imwrite(str(outdir / 'MaxProject_Y' / f'MAX_Y_{stem}'),
                                    my.astype(data_type))

                    state['done'].add((p, t, c))
                    print(f'  P{p} T{t} C{c} ({state["cams"].get(c)}, flip={flip})  '
                          f'max={int(vol3d.max())}  {time.time()-tv:.2f}s')
    finally:
        try:
            with quiet():
                ds.close()
        except Exception:
            pass
    return pending


# ---------------- run ----------------
save_root = Path(SAVE_DIRECTORY)
before = {p.name for p in save_root.glob(f'{SAVE_NAME_PREFIX}*') if p.is_dir()}

t0 = time.time()
lsm.acquisitions().request_run()
print('Acquisition requested...')

run_dir = None
while time.time() - t0 < 120:
    new = {p.name for p in save_root.glob(f'{SAVE_NAME_PREFIX}*') if p.is_dir()} - before
    cand = [save_root / n for n in new if (save_root / n / 'NDTiff.index').exists()]
    if cand:
        run_dir = max(cand, key=lambda p: p.stat().st_mtime)
        break
    time.sleep(POLL_S)

if run_dir is None:
    raise RuntimeError(
        'No new ND-TIFF dataset appeared in 120 s.\n'
        '  - Did the acquisition actually start? Check the LSM window.\n'
        '  - Look in the MM CoreLog for "Expected images did not arrive in '
        'circular buffer" (a camera/trigger failure).\n'
        '  - Confirm the LSM save mode is ND-TIFF and the prefix is '
        f'"{SAVE_NAME_PREFIX}".')
print('Dataset:', run_dir.name)

# Deskewed output lives INSIDE this run's folder, so separate runs never mix.
outdir = run_dir / 'Deskewed'
outdir.mkdir(exist_ok=True)
if Z_max_projections: (outdir / 'MaxProject_Z').mkdir(exist_ok=True)
if Y_max_projections: (outdir / 'MaxProject_Y').mkdir(exist_ok=True)

index_file = run_dir / 'NDTiff.index'
state = {'dsk': None, 'cams': None, 'done': set()}
last_size, last_growth = -1, time.time()

# Phase 1 — wait for the writer to go quiet (only stat() on the index, no reads
# of the image stack, so nothing contends with LSM).
print('Acquiring' + ('  (deskewing live)' if DESKEW_DURING_ACQ else
                     '  (deskew starts when writing finishes)'))
while time.time() - t0 < TIMEOUT_S:
    size = index_file.stat().st_size if index_file.exists() else 0
    if size != last_size:
        last_size, last_growth = size, time.time()
        if DESKEW_DURING_ACQ:
            deskew_pending(run_dir, outdir, state)
    if size > 0 and (time.time() - last_growth) > QUIET_S:
        break
    time.sleep(POLL_S)

acq_s = time.time() - t0
print(f'Writer finished after {acq_s:.1f} s — deskewing...')

# Phase 2 — deskew everything still outstanding
deskew_pending(run_dir, outdir, state)

print(f'\nDone in {time.time()-t0:.1f} s '
      f'(acquire {acq_s:.1f} s) — {len(state["done"])} volume(s) deskewed')
print('Raw     :', run_dir)
print('Deskewed:', outdir)
dsk = state['dsk']
if not state['done']:
    print(f'\nNo volumes deskewed — check SLICES_PER_VIEW ({EXPECTED_NZ}) '
          'matches what LSM acquired.')

## 6. Inspect the last deskewed volume

In [ ]:
v = dsk.deskewed
print('shape:', v.shape, ' dtype:', v.dtype, ' max:', v.max())

fig, ax = plt.subplots(1, 3, figsize=(17, 5))
ax[0].imshow(v[v.shape[0] // 2], cmap='gray');  ax[0].set_title('Mid slice')
ax[1].imshow(dsk.max_z, cmap='gray');           ax[1].set_title('Max projection Z')
ax[2].plot(v.reshape(v.shape[0], -1).max(axis=1), 'o-', ms=3)
ax[2].set_xlabel('slice'); ax[2].set_ylabel('max intensity'); ax[2].grid(True)
ax[2].set_title('Intensity profile')
for a in ax[:2]: a.axis('off')
plt.tight_layout(); plt.show()

## 7. Close

In [ ]:
lsm_mgr.close()
print('LSM closed.')
print('PLogic OutputChannel :', core.get_property('PLogic:E:36', 'OutputChannel'))
print('Auto-shutter         :', core.get_auto_shutter())